# Clean up of redundant rows and columns in the merged Booking and Measurements Dataset

---

In [1]:
import duckdb

In [2]:
# Paths
merged_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements.parquet"
cleaned_output_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_cleaned.parquet"

In [3]:
# Connect to DuckDB
con = duckdb.connect()

In [4]:
# Load merged dataset as a view
con.execute(f"CREATE OR REPLACE VIEW merged AS SELECT * FROM '{merged_file}';")

In [5]:
# Drop exact duplicate rows
con.execute("CREATE OR REPLACE TEMP TABLE merged_dedup AS SELECT DISTINCT * FROM merged;")

In [6]:
# Compute similarity report for numerical/categorical columns (serial_number_id, station_id, book_state)
similarity_report = con.execute("""
    WITH stats AS (
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN serial_number_id = serial_number_id_1 THEN 1 ELSE 0 END) AS serial_match,
            SUM(CASE WHEN station_id = station_id_1 THEN 1 ELSE 0 END) AS station_match,
            SUM(CASE WHEN book_state = book_state_1 THEN 1 ELSE 0 END) AS book_state_match
        FROM merged_dedup
    )
    SELECT
        ROUND(serial_match::DOUBLE / total_rows, 4) AS serial_number_id_match_ratio,
        ROUND(station_match::DOUBLE / total_rows, 4) AS station_id_match_ratio,
        ROUND(book_state_match::DOUBLE / total_rows, 4) AS book_state_match_ratio
    FROM stats;
""").fetchdf()

print("Similarity Report (exact matches):")
print(similarity_report)

Similarity Report (exact matches):
   serial_number_id_match_ratio  station_id_match_ratio  \
0                        0.9999                  0.9999   

   book_state_match_ratio  
0                     1.0  


In [7]:
# Handle datetime approximate matches within tolerance
time_report = con.execute("""
    WITH stats AS (
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN ABS(DATE_DIFF('milliseconds', created_at, created_at_1)) <= 500 THEN 1 ELSE 0 END) AS close_match
        FROM merged_dedup
    )
    SELECT
        ROUND(close_match::DOUBLE / total_rows, 4) AS created_at_close_match_ratio,
        ROUND(1 - (close_match::DOUBLE / total_rows), 4) AS created_at_mismatch_ratio
    FROM stats;
""").fetchdf()

print("Datetime Similarity Report (within ±500ms):")
print(time_report)

Datetime Similarity Report (within ±500ms):
   created_at_close_match_ratio  created_at_mismatch_ratio
0                        0.9969                     0.0031


In [8]:
# Create cleaned table:
con.execute("""
    CREATE OR REPLACE TABLE merged_cleaned AS
    SELECT
        -- Use COALESCE for categorical columns (prefer the left unless NULL)
        COALESCE(serial_number_id, serial_number_id_1) AS serial_number_id,
        COALESCE(station_id, station_id_1) AS station_id,
        COALESCE(book_state, book_state_1) AS book_state,
        -- For datetime, choose created_at if within 500ms, otherwise use created_at_1
        CASE
            WHEN ABS(DATE_DIFF('milliseconds', created_at, created_at_1)) <= 500 THEN created_at
            ELSE created_at_1
        END AS created_at,
        -- Include remaining non-duplicate columns
        *
        EXCLUDE(serial_number_id, serial_number_id_1, station_id, station_id_1, book_state, book_state_1, created_at, created_at_1)
    FROM merged_dedup;
""")

In [9]:
# Save cleaned dataset
con.execute(f"COPY merged_cleaned TO '{cleaned_output_file}' (FORMAT 'parquet');")

In [11]:
# Final summary
shape = con.execute("SELECT COUNT(*) AS rows FROM merged_cleaned;").fetchdf()
print(f"\nCleaned merged file saved to: {cleaned_output_file}")
print(f"Final shape:", shape['rows'][0])

# Print head
head = con.execute("SELECT * FROM merged_cleaned LIMIT 10;").df()
print("\nHead of cleaned dataset:\n", head)


Cleaned merged file saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_cleaned.parquet
Final shape: 43934134

Head of cleaned dataset:
   serial_number_id station_id  book_state                       created_at  \
0         083d7e75   38b291ac           0 2025-05-08 21:00:21.155000+02:00   
1         818749bf   38b291ac           0 2025-05-06 23:15:12.302000+02:00   
2         d8462f9e   38b291ac           0 2025-05-09 10:31:42.424000+02:00   
3         d7df6733   38b291ac           0 2025-04-23 19:41:12.556000+02:00   
4         10752aa5   38b291ac           0 2025-05-06 19:48:44.854000+02:00   
5         5f33e720   38b291ac           0 2025-04-23 16:54:27.719000+02:00   
6         4d8f2759   38b291ac           0 2025-04-17 04:03:17.773000+02:00   
7         c77503b6   38b291ac           0 2025-04-23 22:41:12.808000+02:00   
8         cdca559b   38b291ac           0 2025-04-30 10:00:50.494000+02:00   
9     

In [12]:
con.close()